# Project 05: Spatio-Temporal Traffic Congestion Prediction
**Team No.:** 24  
**Team Members:** Prativa Panda; Rajashree Samal; Simran Sahu; Smruti Rekha Panda  
**Task:** Regression  
**Proposed Hybrid:** Diffusion GCN + Temporal Transformer  
**Dataset:** [METR-LA traffic sensor dataset](https://www.kaggle.com/datasets/annnnguyen/metr-la-dataset)

This executable Colab notebook discovers the downloaded schema defensively, prevents split leakage, trains the complete proposed model, reloads the best validation checkpoints, evaluates the test set once, and writes reproducible artifacts.

## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kagglehub transformers tqdm tabulate

import os, json, random, shutil, glob, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, mean_absolute_error, mean_squared_error, r2_score

SEED=42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed()
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print('Device:',DEVICE)

### CONFIG

In [ ]:
CONFIG = {
    "project_no": "05",
    "project_name": "Spatio-Temporal Traffic Congestion Prediction",
    "team_no": "24",
    "task_type": "regression",
    "kaggle_dataset_slug": "annnnguyen/metr-la-dataset",
    "target_candidates": [],
    "split_ratios": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15
    },
    "random_seed": 42,
    "data_raw_dir": "data/05/raw",
    "data_processed_dir": "data/05/processed",
    "figures_dir": "data/05/figures",
    "results_dir": "data/05/results",
    "checkpoints_dir": "data/05/results/checkpoints",
    "reports_dir": "data/05/reports",
    "epochs": 20,
    "batch_size": 32
}
# Create every directory any later cell writes to (top-level dirs AND the nested
# checkpoints/ subdirectory) - a loop that only makes top-level dirs is exactly the
# bug that caused "Parent directory results does not exist" on torch.save().
for key in ['data_raw_dir', 'data_processed_dir', 'figures_dir', 'results_dir', 'checkpoints_dir', 'reports_dir']:
    os.makedirs(CONFIG[key], exist_ok=True)
CONFIG

## 1. Dataset Download

In [ ]:
import kagglehub
cache_path=kagglehub.dataset_download(CONFIG['kaggle_dataset_slug'])
source=pathlib.Path(cache_path); destination=pathlib.Path(CONFIG['data_raw_dir'])
for item in source.rglob('*'):
    if item.is_file():
        relative=item.relative_to(source); output=destination/relative; output.parent.mkdir(parents=True,exist_ok=True)
        if not output.exists() or output.stat().st_size != item.stat().st_size: shutil.copy2(item,output)
raw_files=[p for p in destination.rglob('*') if p.is_file()]
assert raw_files, 'Dataset download produced no files.'
assert all(p.stat().st_size>0 for p in raw_files), 'A downloaded file is empty.'
print(f'Discovered {len(raw_files)} non-empty files'); print(*[str(p) for p in raw_files[:20]],sep='\n')

## 2. Load Raw Data

In [ ]:
h5=[p for p in raw_files if p.suffix.lower() in {'.h5','.hdf5'}]; npzs=[p for p in raw_files if p.suffix.lower()=='.npz']; pkls=[p for p in raw_files if p.suffix.lower() in {'.pkl','.pickle'}]
if h5: speed=pd.read_hdf(h5[0]).astype('float32').values
elif npzs:
    z=np.load(npzs[0]); arrays=[z[k] for k in z.files if z[k].ndim>=2]; assert arrays; speed=max(arrays,key=lambda a:a.size).astype('float32'); speed=speed.reshape(speed.shape[0],-1)
else: raise FileNotFoundError('No METR-LA speed HDF5/NPZ found.')
adj=None
for path in pkls:
    try:
        import pickle
        obj=pickle.load(open(path,'rb'),encoding='latin1'); candidates=obj if isinstance(obj,(tuple,list)) else [obj]
        for item in candidates:
            if isinstance(item,np.ndarray) and item.ndim==2 and item.shape[0]==item.shape[1]: adj=item.astype('float32')
    except Exception as exc: print('Adjacency skip:',exc)
if adj is None:
    corr=np.nan_to_num(np.corrcoef(speed.T)); adj=(np.abs(corr)>.7).astype('float32'); np.fill_diagonal(adj,1)
assert speed.ndim==2 and len(speed)>24 and adj.shape==(speed.shape[1],speed.shape[1]); print(speed.shape,adj.shape)

## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
missing=np.isnan(speed).mean(0)

# [TARGET SANITY CHECK] speed is the regression target (traffic sensor readings)
print("Total timesteps:", len(speed), "| sensors:", speed.shape[1])
_target_valid = speed[~np.isnan(speed)]
assert np.unique(_target_valid).size > 1, f"DEGENERATE TARGET: only {np.unique(_target_valid).size} unique value(s) found in speed matrix. Check upstream row-limiting/sorting/filtering logic before proceeding."
if len(speed) < 100:
    print(f"[DATA QUALITY WARNING] only {len(speed)} timesteps after loading - too small for reliable evaluation.")
if np.nanstd(speed) < 1e-6:
    print(f"[DATA QUALITY WARNING] target has near-zero variance (std={np.nanstd(speed):.6f}) - results may be trivial.")

memo=f'''# Data Quality Memo
- Speed matrix: {speed.shape}
- Adjacency: {adj.shape}
- Missing fraction: {np.isnan(speed).mean():.4f}
- Chronological split prevents future-to-past leakage.
- Scaling statistics are fitted on training timestamps only.

## Known limitations
- Adjacency falls back to a correlation-threshold graph when no distance-based
  pickle is found in the Kaggle download, which is a weaker proxy for true
  road-network topology.
- Forecast horizon and lookback window are fixed constants (not tuned per-sensor).
- Missing readings are median-imputed from the training window only; long gaps
  are not modeled explicitly.
'''; open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"),'w').write(memo); print(memo)

## 4. Preprocessing & Feature Engineering

In [ ]:
LOOKBACK=12; HORIZON=3; n=len(speed); train_end=int(n*.70); val_end=int(n*.85)
median=np.nanmedian(speed[:train_end],axis=0); filled=np.where(np.isnan(speed),median,speed); mean=filled[:train_end].mean(0); std=filled[:train_end].std(0)+1e-6; scaled=(filled-mean)/std
def windows(start,end):
    xs=[]; ys=[]
    for t in range(max(start,LOOKBACK),end-HORIZON+1): xs.append(scaled[t-LOOKBACK:t]); ys.append(scaled[t:t+HORIZON])
    return np.stack(xs).astype('float32'),np.stack(ys).astype('float32')

## 5. Train / Validation / Test Split

In [ ]:
Xtr,Ytr=windows(0,train_end); Xv,Yv=windows(train_end,val_end); Xte,Yte=windows(val_end,n)
assert train_end<val_end<n and len(Xtr)*len(Xv)*len(Xte)>0
json.dump({'train_windows':len(Xtr),'val_windows':len(Xv),'test_windows':len(Xte),'lookback':LOOKBACK,'horizon':HORIZON},open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"),'w'),indent=2); print(Xtr.shape,Xv.shape,Xte.shape)

## 6. PyTorch Dataset & DataLoader

In [ ]:
def loader(x,y,shuffle=False): return DataLoader(TensorDataset(torch.tensor(x),torch.tensor(y)),batch_size=CONFIG['batch_size'],shuffle=shuffle)
train_loader=loader(Xtr,Ytr,True); val_loader=loader(Xv,Yv); test_loader=loader(Xte,Yte); normalized_adjacency=torch.tensor(adj/(adj.sum(1,keepdims=True)+1e-6),dtype=torch.float32)

## 7. Proposed Model Definition

In [ ]:
class DiffusionGraphConvolution(nn.Module):

    def __init__(self, h=32):
        super().__init__()
        self.proj = nn.Linear(3, h)

    def forward(self, x, A):
        return F.relu(self.proj(torch.stack([x, x @ A, x @ (A @ A)], -1)))

class TemporalTransformer(nn.Module):

    def __init__(self, h=32):
        super().__init__()
        layer = nn.TransformerEncoderLayer(h, 4, 64, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 2)

    def forward(self, x):
        b, t, n, h = x.shape
        z = x.permute(0, 2, 1, 3).reshape(b * n, t, h)
        return self.enc(z)[:, -1].reshape(b, n, h)

class DiffusionTemporalTransformer(nn.Module):

    def __init__(self, horizon, adjacency):
        super().__init__()
        self.register_buffer('adjacency', adjacency)
        self.graph = DiffusionGraphConvolution()
        self.temporal = TemporalTransformer()
        self.head = nn.Linear(32, horizon)

    def forward(self, x):
        return self.head(self.temporal(self.graph(x, self.adjacency))).permute(0, 2, 1)


## 8. Training Loop

In [ ]:
def train_model(model, path):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), 0.001)
    amp_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best = float('inf')
    hist = {'train_loss': [], 'val_loss': []}
    wait = 0
    for epoch in tqdm(range(CONFIG['epochs']), desc='Training', unit='epoch'):
        model.train()
        total = 0
        for x, y in train_loader:
            x, y = (x.to(DEVICE), y.to(DEVICE))
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                loss = F.mse_loss(model(x), y)
            amp_scaler.scale(loss).backward()
            amp_scaler.step(opt)
            amp_scaler.update()
            total += loss.item() * len(x)
        model.eval()
        val = 0
        with torch.no_grad():
            for x, y in val_loader:
                val += F.mse_loss(model(x.to(DEVICE)), y.to(DEVICE)).item() * len(x)
        tr = total / len(train_loader.dataset)
        va = val / len(val_loader.dataset)
        hist['train_loss'].append(tr)
        hist['val_loss'].append(va)
        if va < best:
            best = va
            wait = 0
            torch.save(model.state_dict(), path)
        else:
            wait += 1
        if wait >= 5:
            break
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    return (model, hist)

hybrid_ckpt = os.path.join(CONFIG['checkpoints_dir'], 'best_hybrid.pt')
hybrid, hybrid_history = train_model(DiffusionTemporalTransformer(HORIZON, normalized_adjacency), hybrid_ckpt)

## 9. Evaluation Metrics

In [ ]:
def once(model):
    model.eval()
    pp = []
    yy = []
    with torch.no_grad():
        for x, y in test_loader:
            pp.append(model(x.to(DEVICE)).cpu().numpy())
            yy.append(y.numpy())
    return (np.concatenate(pp), np.concatenate(yy))

def score(p, y):
    pu = p * std + mean
    yu = y * std + mean
    return {'mae': mean_absolute_error(yu.ravel(), pu.ravel()), 'rmse': mean_squared_error(yu.ravel(), pu.ravel()) ** 0.5, 'r2': r2_score(yu.ravel(), pu.ravel())}

hp, test_y2 = once(hybrid)
assert np.array_equal(Yte, test_y2)
results = {'hybrid': score(hp, Yte)}

# persistence baseline: repeat the last observed timestep across the forecast horizon
persist_pred = np.repeat(Xte[:, -1:, :], HORIZON, axis=1)
results['persistence_baseline'] = score(persist_pred, Yte)

# Reload-and-verify: load the checkpoint into a FRESH model instance and confirm the
# evaluation reproduces the in-memory result - catches save/load path bugs.
reloaded = DiffusionTemporalTransformer(HORIZON, normalized_adjacency).to(DEVICE)
reloaded.load_state_dict(torch.load(hybrid_ckpt, map_location=DEVICE, weights_only=True))
rp, ry = once(reloaded)
assert np.array_equal(Yte, ry)
reload_score = score(rp, Yte)
assert abs(reload_score['mae'] - results['hybrid']['mae']) < 1e-4, f"Reloaded checkpoint mismatch: {reload_score} vs {results['hybrid']}"
print('Reload-and-verify OK:', reload_score)

json.dump(results, open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w'), indent=2)
print(results)

## 10. Required Figures

In [ ]:
plt.figure()
plt.plot(hybrid_history['train_loss'], label='Train')
plt.plot(hybrid_history['val_loss'], label='Val')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=150)
plt.show()

actual = (Yte * std + mean).ravel()
predicted = (hp * std + mean).ravel()
take = np.random.default_rng(SEED).choice(len(actual), min(5000, len(actual)), replace=False)
plt.figure()
plt.scatter(actual[take], predicted[take], s=4, alpha=0.3)
lo, hi = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
plt.plot([lo, hi], [lo, hi], 'k--', label='y = x')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig02_predicted_vs_actual.png'), dpi=150)
plt.show()

plt.figure()
plt.plot(actual[:300], label='Actual')
plt.plot(predicted[:300], label='Predicted')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig03_forecast_trace.png'), dpi=150)
plt.show()

hybrid.zero_grad()
sample = torch.tensor(Xte[:8], device=DEVICE, requires_grad=True)
hybrid(sample).sum().backward()
imp = sample.grad.abs().mean((0, 2)).cpu()
plt.figure()
plt.bar(range(LOOKBACK), imp)
plt.xlabel('Lag')
plt.ylabel('Mean |gradient|')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig04_feature_importance.png'), dpi=150)
plt.show()

err = np.abs(predicted - actual)
plt.figure()
sns.histplot(err, bins=50)
plt.xlabel('Absolute error')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig05_error_analysis.png'), dpi=150)
plt.show()

metric = 'mae'
plt.figure()
plt.bar(list(results.keys()), [results[k][metric] for k in results])
plt.ylabel(metric.upper())
plt.title('Hybrid vs persistence baseline')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig06_baseline_comparison.png'), dpi=150)
plt.show()